# Fast-ion distribution

The steady state of a beam is more than its moments: `slowing_down_distribution` returns the fast ions over shell, energy and pitch, $f(\rho, E, \xi) = \mathrm{d}n / (\mathrm{d}E\,\mathrm{d}\xi)$ in $\mathrm{1/(m^3\,eV)}$ with $\xi = v_\parallel / v$, as an xarray dataset.

Each birth slows down on its shell as in `slow_down`, $\mathrm{d}v/\mathrm{d}t = -(v^3 + v_c^3) / (\tau_s v^2)$, which gives the classical slowing-down distribution $f \propto S \tau_s / (v^3 + v_c^3)$ below the birth speed (Stix, Plasma Phys. 14, 367, 1972). Pitch-angle scattering spreads the birth pitch $\xi_0$ over the Legendre polynomials, the eigenfunctions of the Lorentz operator, each decaying at its own rate (Gaffey, J. Plasma Phys. 16, 149, 1976, without his energy diffusion):

$$
f(v, \xi) = \frac{S\,\tau_s}{2\pi (v^3 + v_c^3)} \sum_l \frac{2l + 1}{2} P_l(\xi_0) P_l(\xi) \left[\frac{v^3 (v_0^3 + v_c^3)}{v_0^3 (v^3 + v_c^3)}\right]^{l(l+1)\beta/6},
$$

with $\beta = Z_\mathrm{eff} / [Z]$. freeofcharge evaluates it with each species' own Coulomb logarithm, as `slow_down` does, and averages it over energy and pitch bins.

In [ ]:
import numpy as np
import plasma_plots as pp
import xarray as xr
from plasmacoll.constants import ATOMIC_MASS, ELEMENTARY_CHARGE

import freeofcharge as foc
from freeofcharge import orbits
from freeofcharge.distribution import (
    distribution_at,
    distribution_moments,
    energy_spectrum,
    orbit_distribution,
    pitch_distribution,
    slowing_down_distribution,
)
from freeofcharge.slowing_down import critical_velocity

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

## A beam with three energy components

The tokamak of "Heating and current drive", with a cooler and thinner plasma, and a positive-ion deuterium source at 120 keV whose molecular ions give atoms at a half and a third of the energy too. Its power shares (70, 20 and 10 %) are illustrative. Each component is deposited and turned into a distribution on the same energy bins of 1 keV, so 60 and 40 keV fall on bin edges.

In [ ]:
equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
plasma = foc.Plasma.parabolic(5e19, 8e3, ions={"D": (2, 1, 0.96), "He": (4, 2, 0.02)})
rho_edges = np.linspace(0.0, 1.0, 11)
surfaces = equilibrium.flux_surfaces(rho_edges)
origin, direction = foc.tangential_injection(5.3, 12.0)
source = foc.BeamSource(
    "NB",
    origin,
    direction,
    120e3,
    10e6,
    power_fractions=(0.7, 0.2, 0.1),
    mass_number=2,
    width=0.2,
    divergence=0.01,
)
energy_edges = np.linspace(0.0, 120e3, 121)
parts = {}
for name, beam in source.beams(markers=1500, seed=1).items():
    deposition = foc.deposit(
        beam,
        equilibrium,
        plasma,
        step=0.03,
        births_per_marker=2,
        rho_edges=rho_edges,
        seed=2,
    )
    parts[name] = slowing_down_distribution(
        deposition, plasma, surfaces, energy_edges=energy_edges
    )
    # the moments of f are those of slow_down with the local pitch
    local = foc.slow_down(deposition, plasma, surfaces, orbit_average=False)
    moments = distribution_moments(parts[name])
    for key in ("fast_ion_density", "fast_ion_pressure", "fast_ion_current_density"):
        error = np.max(np.abs(moments[key].values / getattr(local, key) - 1))
        assert error < 0.01, (name, key, error)
    print(
        f"{name:>5}: {deposition.absorbed_power / 1e6:.1f} MW absorbed, "
        f"{local.stored_energy / 1e6:.2f} MJ in fast ions"
    )

component = xr.DataArray(list(parts), dims="component", name="component")
distribution = xr.concat(list(parts.values()), dim=component, data_vars=["f"])
distribution

The density, pressure and current of each component's $f$ agree with `slow_down(..., orbit_average=False)` to better than 1 % on every shell (checked above). The dataset holds `f` over `(component, rho, energy, pitch)`, the shell volumes and the bin edges.

## $f(E, \xi)$ on three shells

The sum of the components on three shells, on a logarithmic colour scale from $10^9$ to $10^{13}\,\mathrm{m^{-3}\,eV^{-1}}$. The ions are born at the pitch where the tangential beam crosses the field, near $\xi_0 \approx 0.75$ on the inner shell and lower farther out, and spread in pitch as they slow down: the farther below the birth energy, the wider. On the outer shells the half and third energy components show as steps at 60 and 40 keV.

In [ ]:
def log_map(f):
    # below 1e9 / (m^3 eV), with the slight ringing of the series, at the floor
    out = np.log10(np.maximum(f, 1e9)).assign_coords(energy=f.energy / 1e3)
    out.energy.attrs = {"long_name": "$E$", "units": "keV"}
    out.attrs = {"long_name": r"$\log_{10} f$"}
    return out


total = distribution.sum("component").assign(volume=distribution.volume)
total = total.assign_attrs(distribution.attrs)
with pp.figure(1, 3, figsize=(12, 3.8), sharey=True) as fig:
    for i, rho in enumerate((0.25, 0.55, 0.85)):
        at = log_map(distribution_at(total, rho))
        at.plasma.plot.slice(
            x="energy",
            y="pitch",
            vmin=9,
            vmax=13,
            title=f"rho = {float(at.rho):.2f}",
            colorbar_label=r"$\log_{10} f$",
            ax=fig[i],
        )

The three components on the middle shell: each starts at its own birth energy. At the same energy the lower components are less spread in pitch, having been born closer to it.

In [ ]:
with pp.figure(1, 3, figsize=(12, 3.8), sharey=True) as fig:
    for i, name in enumerate(distribution.component.values):
        at = log_map(distribution_at(distribution.sel(component=name), 0.55))
        at.plasma.plot.slice(
            x="energy",
            y="pitch",
            vmin=9,
            vmax=13,
            title=f"{name}, rho = {float(at.rho):.2f}",
            colorbar_label=r"$\log_{10} f$",
            ax=fig[i],
        )

## The energy spectrum

Integrated over the pitch, $\mathrm{d}n/\mathrm{d}E = 4\pi v^2 f\,\mathrm{d}v/\mathrm{d}E \propto v / (v^3 + v_c^3)$ for each component below its birth energy: falling as $1/E$ well above the critical energy $E_c$, where the electrons drag, rising as $\sqrt{E}$ well below it, where the ions do, with its maximum at $2^{-2/3} E_c \approx 0.63\,E_c$. The dotted lines are this shape with the critical velocity of equal Coulomb logarithms (`critical_velocity`), each scaled to the mean of its component.

In [ ]:
rho_shell = 0.55
spectra = energy_spectrum(distribution).sel(rho=rho_shell, method="nearest")
spectra = spectra.assign_coords(energy=spectra.energy / 1e3)
spectra.energy.attrs = {"long_name": "$E$", "units": "keV"}
v_c = float(
    critical_velocity(
        plasma.electron_density(rho_shell),
        plasma.electron_temperature(rho_shell),
        plasma.ion_densities(rho_shell),
        plasma.ion_masses,
        plasma.ion_charges,
    )
)
mass = 2 * ATOMIC_MASS
e_c = 0.5 * mass * v_c**2 / ELEMENTARY_CHARGE
speed = np.sqrt(2 * spectra.energy.values * 1e3 * ELEMENTARY_CHARGE / mass)
shape = speed / (speed**3 + v_c**3)

with pp.figure(figsize=(6.5, 4)) as fig:
    spectra.plasma.plot.profiles(x="energy", over="component", title="", ax=fig[0])
    for name, top in zip(spectra.component.values, (120, 60, 40), strict=True):
        below = spectra.energy.values < top
        weight = spectra.sel(component=name).values[below][:-1].mean()
        fig[0].plot(
            spectra.energy.values[below],
            shape[below] * weight / shape[below][:-1].mean(),
            color="0.4",
            linestyle=":",
        )
    fig[0].axvline(e_c / 1e3, color="0.6", linestyle="--", label="$E_c$")
    fig[0].legend(fontsize="small")

ratio = spectra.sel(component="full").values[:-1] / shape[:-1]
ratio /= ratio.mean()
print(f"rho = {float(spectra.rho):.2f}: E_c = {e_c / 1e3:.0f} keV")
print(
    f"full component over the shape: {ratio.min():.3f} to {ratio.max():.3f} of its mean"
)

The computed spectra follow the classical shape to a few per cent (printed above). The rest comes from the Coulomb logarithms: `critical_velocity` takes equal ones, while the distribution uses each species' own, the ion ones growing with the fast ion's speed.

## Anisotropy

A tangential beam builds a parallel pressure: `distribution_moments` splits the fast-ion pressure into $p_\parallel = \int m v^2 \xi^2 f$ and $p_\perp = \int \tfrac12 m v^2 (1 - \xi^2) f$, with $p = (p_\parallel + 2 p_\perp)/3$ as `slow_down`'s isotropic estimate, and gives the current density and the ratio $p_\perp / p_\parallel$, which is 1 for an isotropic distribution.

In [ ]:
moments = distribution_moments(total)
pressures = (
    moments[["parallel_pressure", "perpendicular_pressure"]]
    .rename(
        {"parallel_pressure": "parallel", "perpendicular_pressure": "perpendicular"}
    )
    .to_array("direction", name="fast-ion pressure")
)
pressures.attrs["units"] = "Pa"
anisotropy = moments.anisotropy.values
print("p_perp / p_par from the centre out:", np.round(anisotropy, 2))
with pp.figure(1, 2, figsize=(10, 3.8)) as fig:
    pressures.plasma.plot.profiles(x="rho", over="direction", title="", ax=fig[0])
    moments.anisotropy.plasma.plot.lineout(x="rho", step="mid", title="", ax=fig[1])

Near the axis, where the beam runs closest to the field, $p_\perp / p_\parallel$ is under 0.4; farther out the beam crosses the field at a larger angle, and on the outermost shell the ratio passes 1.

## Against the orbits

`follow_orbits` records the same distribution from the guiding-centre orbits with collisions when given `histogram=(energy_edges, pitch_edges)`: each marker's time in each shell, energy and pitch bin, times its weight. `orbit_distribution` turns it into this layout. Narrow orbits need a high field: a hydrogen beam at 30 keV in 10 T, without energy diffusion (the local model has none). At an aspect ratio of 17 the field barely varies on a flux surface, and the two agree:

In [ ]:
def compare(major_radius, markers=600):
    eq = foc.Equilibrium.solovev(major_radius, 0.6, 1.5, 10.0, 1.2, wall_psi_n=None)
    edges = np.linspace(0, 1, 6)
    shells = eq.flux_surfaces(edges)
    field = orbits.orbit_field(eq, (96, 4, 96))
    hot = foc.Plasma.parabolic(1e20, 600.0, ions={"H": (1, 1, 1.0)})
    start, aim = foc.tangential_injection(major_radius - 0.1, major_radius + 3.0)
    beam = foc.NeutralBeam.gaussian(
        start,
        aim,
        30e3,
        1e6,
        mass_number=1,
        width=0.1,
        divergence=0.005,
        markers=markers,
        seed=1,
    )
    births = foc.deposit(beam, eq, hot, step=0.01, rho_edges=edges, seed=2)
    bins = (np.linspace(0, 30e3, 31), np.linspace(-1, 1, 21))
    local = slowing_down_distribution(
        births, hot, shells, energy_edges=bins[0], pitch_bins=20, thermal_factor=2.0
    )
    result = orbits.follow_orbits(
        births,
        field,
        hot,
        surfaces=shells,
        step=0.2,
        thermal_factor=2.0,
        energy_diffusion=False,
        seed=1,
        histogram=bins,
    )
    traced = orbit_distribution(result, mass_number=1)
    return xr.concat(
        [local, traced],
        dim=xr.DataArray(["local", "orbits"], dims="model", name="model"),
        data_vars=["f"],
    )


def totals(both):
    moments = distribution_moments(both)
    keys = [
        "fast_ion_density",
        "fast_ion_pressure",
        "fast_ion_current_density",
        "parallel_pressure",
        "perpendicular_pressure",
    ]
    content = (moments[keys] * both.volume).sum("rho")
    return {
        key: float(content[key].sel(model="orbits") / content[key].sel(model="local"))
        for key in keys
    }


narrow = compare(10.0)
for key, value in totals(narrow).items():
    print(f"{key:>25}: orbits / local = {value:.3f}")

The ion content, pressure, parallel pressure and current from the orbits are within 2 % of the local model, the perpendicular pressure within 6 % (about 1 % of the markers leave the plasma at $\rho = 1$, which the local model does not know). The spectra and the pitch distributions agree on the shell with most ions, up to the sampling noise of 600 markers:

In [ ]:
spectra = energy_spectrum(narrow).sel(rho=0.7, method="nearest")
spectra = spectra.assign_coords(energy=spectra.energy / 1e3)
spectra.energy.attrs = {"long_name": "$E$", "units": "keV"}
pitches = pitch_distribution(narrow).sel(rho=0.7, method="nearest")
with pp.figure(1, 2, figsize=(10, 3.8)) as fig:
    spectra.plasma.plot.profiles(
        x="energy", over="model", title=f"rho = {float(spectra.rho):.1f}", ax=fig[0]
    )
    pitches.plasma.plot.profiles(
        x="pitch", over="model", title=f"rho = {float(pitches.rho):.1f}", ax=fig[1]
    )

At an aspect ratio of 5 the field on a flux surface varies by $\pm 20$ %: the local pitch of an ion changes along its orbit (the mirror force) and those with a small pitch are trapped, bouncing between both signs of $\xi$. The local model keeps the birth pitch, scattered as in a uniform field:

In [ ]:
wide = compare(3.0)
for key, value in totals(wide).items():
    print(f"{key:>25}: orbits / local = {value:.3f}")
pitch = pitch_distribution(wide)
content = (pitch * wide.volume).sum("rho")
negative = content.where(content.pitch < 0).sum("pitch") / content.sum("pitch")
for model in ("local", "orbits"):
    print(
        f"{model}: {100 * float(negative.sel(model=model)):.1f} % of the ions at negative pitch"
    )
content.attrs = {"long_name": r"$\mathrm{d}N/\mathrm{d}\xi$", "units": "1"}
content.plasma.plot.profiles(
    x="pitch", over="model", title="aspect ratio 5, all shells"
)

The ion content still agrees, but the orbits put more ions at negative pitch and fewer near $\xi = 1$: their perpendicular pressure is about 12 % higher and their current 8 % lower. This is the trapping that the orbit average of `slow_down` (`orbit_average=True`, `freeofcharge.trapping`) takes into account for the current, and the local $f$ leaves out.

What the local distribution leaves out: the orbit width and the mirror force (every ion stays on its birth shell at its local pitch), energy diffusion (nothing above the birth energy, nothing below the end energy), losses and charge exchange. The Legendre series is cut after 48 terms (`legendre_modes`): just below the birth energy, where the pitch is still narrow, the bins can ring slightly below zero; the moments in $\xi$ and $\xi^2$ do not depend on the cut.